# Wind tutorial 2

In [ ]:
import enn554.wind as wind
import scipy.stats as sps
from scipy.special import gamma
import matplotlib.pyplot as plt
from enn554.paths import data_dir
import numpy as np
import pandas as pd
dd = data_dir()

## Exercise 1
I'm going to to this in Excel in class to give it a more hands-on feel. I'll use the toolbox here to verify

In [ ]:
gamma(1.5), np.sqrt(np.pi)/2

In [ ]:
scale = 10.0/gamma(1.5)/np.sqrt(2) # weird sqrt(2) here because of SciPy's strange parameterization
ws_dist = sps.rayleigh(scale=scale)
rho_air = 1.2
Area = np.pi/4.0 * 98**2
def power_curve(u):
    if (u < 4.0) or (u>22):
        return 0
    elif u<=14:
        return 0.38/2*rho_air*Area*u**3/1e6
    else:
        return 0.38/2*rho_air*Area*14**3/1e6
    
fig,ax = plt.subplots()
w = np.array([ii for ii in range(23)])
# w = np.linspace(0,30,1000)
ax.plot(w,[power_curve(ws) for ws in w])
ax.set_xlabel('Wind speed (m/s)')
ax.set_ylabel('Power (MW)')


F = ws_dist.cdf(w)
hours = np.diff(F)*8760
wc = w[0:-1] + 0.5*(w[1::]-w[0:-1]) # centers
sum(hours * np.array([power_curve(w) for w in wc]))


In [ ]:
np.diff(F)

## Exercise 2

The axial induction for the back turbine is set to the optimum $a_b=\frac{1}{3}$ since it doesn't have any downstream turbines. The front axial induction $a_f$ will be varied. The power produced by the first turbine is 
    
\begin{align}
    P_{turbine,f} & = \frac{1}{2}C_p\rho A u_{\infty}^3 \\
    & = \frac{1}{2}4a_f(1-a_f)^2\rho A u_{\infty}^3
\end{align}
and
\begin{align}
    \frac{P_{turbine,f}}{P_{turbine,max}} & = \frac{4a_f(1-a_f)^2}{\frac{16}{27}}
\end{align}
The wind speed deficit at the rear turbine is calculated using the Park model
\begin{align}
    \frac{u(x)}{u_\infty} & = 1-2a_f\left(\frac{r_0}{r_0+\alpha \,\Delta d}\right)^2  \\
    & = 1-2a_f\left(\frac{D}{D+2\alpha \,\Delta d \sqrt{\frac{1-2a_f}{1-a_f}}}\right)^2 \\ 
    & = 1-2a_f\left(\frac{1}{1+2\alpha \,\frac{\Delta d}{D}\sqrt{\frac{1-2a_f}{1-a_f}}}\right)^2
\end{align}
where $r_0 = R\sqrt{\frac{1-a}{1-2a}}$. Without loss of generality, assume that the front turbine position is $x=0$. The power produced by the back turbine is then
\begin{align}
    P_{turbine,b} & = \frac{1}{2}4a_b(1-a_b)^2\rho A \; \left( \frac{u(\Delta d)}{u_\infty} \right)^3 u_\infty^3
\end{align}

The back turbine will use $a_b=\frac{1}{3}$ (why?), so $4a_b (1-a_b)^2 = \frac{16}{27}$. We note that 

\begin{align}
    \frac{P_{turbine,b}}{P_{turbine,max}} & = \left( \frac{u(\Delta d)}{u_\infty} \right)^3 \\ 
    & = \left( 1-2a_f\left( \frac{1}{1+2\alpha \,\frac{\Delta d}{D} \sqrt{\frac{1-2a}{1-a}}}\right)^2 \right)^3
\end{align}

In [ ]:
α,Δd = 0.1,20
fig,ax = plt.subplots(figsize=(7,7))
af = np.linspace(0,0.5,1000)
front_norm_power = 27*4*af*(1-af)**2/16.0
factor = np.sqrt((1-2*af)/(1-af))
rear_norm_power = (1-2*af*(1/(1.0+2*α*Δd*factor))**2)**3
total_norm_power = rear_norm_power+front_norm_power

ax.plot(af,front_norm_power,label="Front")
ax.plot(af,rear_norm_power,ls='--',label="Rear")
ax.plot(af,total_norm_power,label="Total")


ax.set_ylim((0,2.5))
ax.set_xlim((0,0.5))
ax.set_ylabel('Normalize power (single turbine max = 1.0)')
ax.set_xlabel('Fron turbine axial induction')

idx_betz = np.argmin( np.abs(af-1.0/3.0) )
idx_best = np.argmax(total_norm_power)

ax.set_title(rf'$\Delta d$ = {Δd:.1f}, $a_f^*=${af[idx_best]:.3f}')
ax.plot([1.0/3.0,1.0/3.0],[0,front_norm_power[idx_betz]],
        ls='--',color='blue',label='Betz optimal (single)',alpha=0.2)
ax.plot([af[idx_best],af[idx_best]],[0,total_norm_power[idx_best]],
        ls='--',color='green',label='Optimal (total)',alpha=0.2)
ax.axhline(2.0,ls=':',color='black',label='Ideal total')

ax.legend(loc='upper left')